# Glass crack models: SDSU TIDE session

**Spawn options** (https://jupyterhub-research.sdsu.edu): image **PyTorch Notebook**, **1 GPU**, 8 CPU cores, 32 GB RAM. Free GPUs right now: https://nrp.ai/viz/resources/ (filter the name column to `rci-tide`).

Everything lives under `~/glass_crack/` in the persistent home `/home/jovyan` (75 GiB): `cracked/` code, `bundle/` data, `venv/` packages, `cache/` pretrained weights, `outputs/` runs, evals, logs and `results/results.csv`. Everything outside `/home/jovyan` (apt/pip/conda installs in the default environment) reverts when the server stops or restarts.

**Idle shutdown:** the server stops after **12 hours with no browser connected** to the hub. Background jobs keep running with the tab closed, as long as you reconnect within 12 hours. Runs longer than that belong in a TIDE batch job (https://csu-tide.github.io/batch-jobs/).

**First time only**
1. Upload `bundle_colab.zip` (185 MB) to `~/glass_crack/bundle_colab.zip` with the JupyterLab upload button (create the folder first).
2. Run cell 1. If the repo is private, clone it once from a JupyterLab **Terminal** instead (GitHub username + personal access token).

**Every session**: cells 1-2, then start jobs. Training runs in the background (`scripts/tide/job.sh`), so closing the tab or restarting the kernel does not stop it. If the server itself stops (12 h idle, manual stop), start the same job again: it resumes from the last checkpoint.

In [ ]:
%%bash
# 1. Code: clone once, then fast-forward each session
REPO_URL=https://github.com/Kuahoo/glass_cracks.git
mkdir -p ~/glass_crack && cd ~/glass_crack
if [ -d cracked/.git ]; then git -C cracked pull --ff-only; else git clone "$REPO_URL" cracked; fi
git -C cracked log -1 --oneline

In [ ]:
%%bash
# 2. Environment (venv reusing the image's CUDA torch + pins), data unzip on first run, relabel, session report
bash ~/glass_crack/cracked/scripts/tide/setup.sh

## Optional checks (first session only)
Unit tests and a 1-epoch GPU smoke run (results go to `results_smoke.csv`, not the main table).

In [ ]:
%%bash
source ~/glass_crack/cracked/scripts/tide/env.sh && cd "$CODE_DIR"
"$PY" -m unittest discover -s glass_crack_models/tests -t . 2>&1 | tail -3
"$PY" -m glass_crack_models.train.train_yolo --config e2_yolo_linear.yaml --smoke --device 0 2>&1 | tail -5

## Train (background jobs)
Each job trains, then evaluates `best` once (threshold frozen on valid, test, NCUGD, AGDD, galleries) and appends a row to `outputs/results/results.csv`. One GPU per server, so start one job at a time and check `status` before the next.

In [ ]:
%%bash
# E2: YOLO-seg, linear_defect
source ~/glass_crack/cracked/scripts/tide/env.sh
bash "$CODE_DIR/scripts/tide/job.sh" start E2_s0 glass_crack_models.train.train_yolo \
    --config e2_yolo_linear.yaml --seed 0 --device 0 --workers "$WORKERS" --compute-units "TIDE $GPU_NAME"

In [ ]:
%%bash
# E1: YOLO-seg, 4 classes (scored as crack OR scratch)
source ~/glass_crack/cracked/scripts/tide/env.sh
bash "$CODE_DIR/scripts/tide/job.sh" start E1_s0 glass_crack_models.train.train_yolo \
    --config e1_yolo_4class.yaml --seed 0 --device 0 --workers "$WORKERS" --compute-units "TIDE $GPU_NAME"

In [ ]:
%%bash
# E3: U-Net ResNet34, linear_defect, BCE + Dice (ablations: e3_unet_r34_tversky.yaml, e3_unet_r34_cldice.yaml)
source ~/glass_crack/cracked/scripts/tide/env.sh
bash "$CODE_DIR/scripts/tide/job.sh" start E3_s0 glass_crack_models.train.train_semseg \
    --config e3_unet_r34.yaml --seed 0 --device cuda --workers "$WORKERS" --compute-units "TIDE $GPU_NAME"

## Monitor

In [ ]:
%%bash
J=~/glass_crack/cracked/scripts/tide/job.sh
bash $J status
echo; bash $J tail E2_s0 20        # change the job name; `bash $J stop E2_s0` stops a job

## Re-score, predict, inspect

In [ ]:
%%bash
source ~/glass_crack/cracked/scripts/tide/env.sh
J="$CODE_DIR/scripts/tide/job.sh"
# Re-score an existing checkpoint from its cached predictions (no retraining):
# bash $J start rescore_E2 glass_crack_models.eval.run_eval --checkpoint "$OUT_ROOT/runs/E2_s0/weights/best.pt" --exp-id E2 --seed 0 --reuse-cache --notes rescored
# Predict on a folder of photos (threshold from operating_point.json next to the checkpoint):
# bash $J start predict_E2 glass_crack_models.infer.predict --checkpoint "$OUT_ROOT/runs/E2_s0/weights/best.pt" --input ~/photos --out "$OUT_ROOT/predictions/E2_s0"
du -sh "$GC_ROOT"/* 2>/dev/null

In [ ]:
import os, pandas as pd
p = os.path.expanduser('~/glass_crack/outputs/results/results.csv')
cols = ['exp_id', 'task', 'seed', 'threshold', 'test_recall', 'test_recall_lo', 'test_recall_hi', 'test_precision',
        'test_mask_iou', 'test_fp100', 'ncugd_recall', 'agdd_pair_recall', 'compute_units', 'notes']
df = pd.read_csv(p) if os.path.exists(p) else None
df[[c for c in cols if c in df.columns]] if df is not None else 'no results yet'